In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# 1. Configuration d'une seed pour avoir les même données pour les 4 modèles d'entraînements
random.seed(42)
np.random.seed(42)

# 2. Nettoyage si nécessaire
df = pd.read_csv('combine_UNSW_test2.csv') 
df['attack_cat'] = df['attack_cat'].str.strip().replace({
    'Backdoors': 'Backdoor', 'Fuzzers ': 'Fuzzers',
    ' Reconnaissance ': 'Reconnaissance', ' Shellcode ': 'Shellcode'
})

X = df.drop(columns=["attack_cat", "label"], errors='ignore')
y = df["attack_cat"]

# 3. Split 70% 15% 15%
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# 4. Prétraitement
cat_cols = ['proto', 'state', 'service']
num_cols = [c for c in X.columns if c not in cat_cols]

preprocessor = ColumnTransformer([
    ("onehot", OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ("scaler", StandardScaler(), num_cols)
])

#  5. PIPELINE AVEC SMOTE & PARAMÈTRES OPTIMISÉS
model_pipeline = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)), 
    ("model", RandomForestClassifier(
        n_estimators=350,       
        max_depth=15,           # LIMITE ajoutée pour éviter le sur-apprentissage (overfitting)
        min_samples_leaf=5,     # Empêche de créer des feuilles pour 1 seul individu
        random_state=42, 
        n_jobs=-1,
        class_weight='balanced_subsample'
    ))
])

# 6. Training & Eval
print("Entraînement en cours (n_estimators=350, max_depth=15)...")
model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_val)
print(classification_report(y_val, y_pred))

Entraînement en cours (n_estimators=350, max_depth=15)...


In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# 1. Binarisation des labels pour le multi-classe
classes = list(model_pipeline.classes_)
y_val_bin = label_binarize(y_val, classes=classes)
n_classes = len(classes)

# 2. Obtenir les probabilités de prédiction
# On récupère les probabilités pour chaque classe
y_score = model_pipeline.predict_proba(X_val)

# 3. Création des graphiques
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

colors = plt.cm.get_cmap('tab10')(np.linspace(0, 1, n_classes))

for i, color in zip(range(n_classes), colors):
    #Courbe ROC
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color=color, lw=2,
             label=f'{classes[i]} (AUC = {roc_auc:.2f})')

    #Courbe Precision-Recall
    precision, recall, _ = precision_recall_curve(y_val_bin[:, i], y_score[:, i])
    avg_prec = average_precision_score(y_val_bin[:, i], y_score[:, i])
    ax2.plot(recall, precision, color=color, lw=2,
             label=f'{classes[i]} (AP = {avg_prec:.2f})')

# Configuration Graphique 1 (ROC)
ax1.plot([0, 1], [0, 1], 'k--', lw=2)
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('Taux de Faux Positifs (FPR)')
ax1.set_ylabel('Taux de Vrais Positifs (TPR / Recall)')
ax1.set_title('Courbes ROC (One-vs-Rest)')
ax1.legend(loc="lower right", fontsize='small')
ax1.grid(alpha=0.3)

# Configuration Graphique 2 (PR)
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Rappel (Recall)')
ax2.set_ylabel('Précision')
ax2.set_title('Courbes Precision-Recall')
ax2.legend(loc="lower left", fontsize='small')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

#1. Configuration de la seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

#2. Nettoyage si nécessaire
df = pd.read_csv('combine_UNSW_test2.csv')
df['attack_cat'] = df['attack_cat'].str.strip().replace({
    'Backdoors': 'Backdoor',
    'Fuzzers ': 'Fuzzers',
    ' Reconnaissance ': 'Reconnaissance',
    ' Shellcode ': 'Shellcode'
})

target = "attack_cat"
X = df.drop(columns=[target, "label"], errors='ignore')
y = df[target]

# Encodage de la cible (Obligatoire pour XGBoost)
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

# 3. Split 70% 15% 15%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded
)

X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Total lignes : {len(df)}")
print(f"Train : {len(X_train)} | Test : {len(X_test)} | Val : {len(X_val)}")

# 4. Prétraitement
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ("onehot", OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ("scaler", StandardScaler(), num_cols)
])

# Application du prétraitement
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
X_val_proc = preprocessor.transform(X_val)

# 5.Calcul des poids
# XGBoost a besoin de poids par échantillon pour équilibrer l'importance des classes rares
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
weights_map = dict(zip(np.unique(y_train), class_weights))
sample_weights = np.array([weights_map[label] for label in y_train])

# 6. Modèle XGBoost
xgb_model = XGBClassifier(
    n_estimators=500,           
    learning_rate=0.05,         # Plus lent pour une meilleure convergence
    max_depth=8,                # Profondeur augmentée pour compenser le manque de SMOTE
    subsample=0.8,              # Utilise 80% des données pour chaque arbre (anti-overfitting)
    colsample_bytree=0.8,       # Utilise 80% des features par arbre
    gamma=1,                    # Régularisation L0 (évite les divisions inutiles)
    objective='multi:softprob',
    random_state=42,
    tree_method='hist',         # Indispensable pour la rapidité sur ce volume
    device="cpu"                
)

#7. Entraînement
print("\nEntraînement de XGBoost avec régularisation et poids de classes...")
xgb_model.fit(
    X_train_proc, y_train, 
    sample_weight=sample_weights,
    eval_set=[(X_val_proc, y_val)], 
    verbose=50 # Affiche l'avancement tous les 50 arbres
)

#8. Évaluer
y_pred_val = xgb_model.predict(X_val_proc)

print("\n=== XGBoost : Rapport de Classification (VALIDATION SET) ===")
print(classification_report(y_val, y_pred_val, target_names=le_target.classes_))

# --- 9. MATRICE DE CONFUSION ---
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_val, y_pred_val)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=le_target.classes_, 
            yticklabels=le_target.classes_)
plt.title('Matrice de Confusion XGBoost')
plt.ylabel('Vrais Labels')
plt.xlabel('Prédictions')
plt.xticks(rotation=45)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# 1. Préparation des données pour le multi-classe
# On binarise y_val
classes_names = le_target.classes_
n_classes = len(classes_names)
y_val_bin = label_binarize(y_val, classes=range(n_classes))

# 2. Obtenir les probabilités de prédiction avec XGBoost
y_score = xgb_model.predict_proba(X_val_proc)

# 3. Création des graphiques
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
colors = plt.cm.get_cmap('tab10')(np.linspace(0, 1, n_classes))

for i, color in zip(range(n_classes), colors):
    # Courbe ROC 
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color=color, lw=2,
             label=f'{classes_names[i]} (AUC = {roc_auc:.2f})')

    # Courbe Precision-Recall
    precision, recall, _ = precision_recall_curve(y_val_bin[:, i], y_score[:, i])
    avg_prec = average_precision_score(y_val_bin[:, i], y_score[:, i])
    ax2.plot(recall, precision, color=color, lw=2,
             label=f'{classes_names[i]} (AP = {avg_prec:.2f})')

# Configuration de la courbe ROC
ax1.plot([0, 1], [0, 1], 'k--', lw=2)
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('Taux de Faux Positifs (FPR)')
ax1.set_ylabel('Taux de Vrais Positifs (TPR / Recall)')
ax1.set_title('Courbes ROC - XGBoost (One-vs-Rest)')
ax1.legend(loc="lower right", fontsize='small')
ax1.grid(alpha=0.3)

# Configuration de la courbe Precision-Recall
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Rappel (Recall)')
ax2.set_ylabel('Précision')
ax2.set_title('Courbes Precision-Recall - XGBoost')
ax2.legend(loc="lower left", fontsize='small')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import torch
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# On utilise pytorch_tabnet (assure-toi d'avoir fait: pip install pytorch-tabnet)
from pytorch_tabnet.tab_model import TabNetClassifier

#1. Configuration de la seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

#2. Nettoyage si nécessaire
df = pd.read_csv('combine_UNSW_test2.csv')
df['attack_cat'] = df['attack_cat'].str.strip().replace({
    'Backdoors': 'Backdoor',
    'Fuzzers ': 'Fuzzers',
    ' Reconnaissance ': 'Reconnaissance',
    ' Shellcode ': 'Shellcode'
})

target = "attack_cat"
X = df.drop(columns=[target, "label"], errors='ignore')
y = df[target]

# Encodage de la cible
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

# 3. Split 70% 15% 15%
# Utilisation de .copy() pour éviter le SettingWithCopyWarning
X_train_raw, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded
)
X_test_raw, X_val_raw, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

X_train = X_train_raw.copy()
X_test = X_test_raw.copy()
X_val = X_val_raw.copy()

# 4. Prétraitement
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# A. Normalisation des numériques
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])

# B. Label Encoding des catégorielles et Embeddings
cat_idxs = []
cat_dims = []

for i, col in enumerate(X.columns):
    if col in cat_cols:
        le = LabelEncoder()
        # Fit sur Train uniquement pour la rigueur, transformation sécurisée pour les autres
        X_train[col] = le.fit_transform(X_train[col].astype(str))
        
        # Gestion des catégories inconnues pour Val et Test
        X_val[col] = X_val[col].astype(str).map(lambda s: le.transform([s])[0] if s in le.classes_ else 0)
        X_test[col] = X_test[col].astype(str).map(lambda s: le.transform([s])[0] if s in le.classes_ else 0)
        
        cat_idxs.append(i)
        cat_dims.append(len(le.classes_))

# Conversion en Numpy
X_train_np = X_train.values
X_test_np = X_test.values
X_val_np = X_val.values

# 5. Cailcul des poids
weights = compute_sample_weight(class_weight='balanced', y=y_train)

# 6. Configuration du modèle
tabnet = TabNetClassifier(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=2,               # Dimension réduite pour éviter le sur-apprentissage
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size":10, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='sparsemax',       # Plus efficace que entmax pour la sélection de features
    seed=42,
    device_name='auto'
)

# 7. Entraînement
print(f"\nEntraînement de TabNet (Deep Learning) en cours...")
tabnet.fit(
    X_train_np, y_train,
    eval_set=[(X_val_np, y_val)],
    patience=15,                 # Un peu plus de patience pour les réseaux de neurones
    batch_size=4096, 
    virtual_batch_size=256,
    max_epochs=100,
    weights=1                    # TabNet gère mieux l'équilibrage via le paramètre weights=1 (balanced interne)
)

# 8. Évaliation
y_pred = tabnet.predict(X_test_np)

print("\n=== TabNet : Rapport Final (TEST SET) ===")
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

# 9. Visualisation de l'attention
# TabNet permet de voir quelles colonnes ont été les plus importantes
feat_importances = tabnet.feature_importances_
indices = np.argsort(feat_importances)[-10:] # Top 10

plt.figure(figsize=(10, 6))
plt.title("Importance des caractéristiques (Top 10) - TabNet")
plt.barh(range(len(indices)), feat_importances[indices], align="center")
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.xlabel("Importance relative")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# 1. Binarisation des labels (nécessaire pour le multi-classe)
classes_names = le_target.classes_
n_classes = len(classes_names)
y_test_bin = label_binarize(y_test, classes=range(n_classes))

# 2. Récupération des probabilités de prédiction sur le set de TEST
y_score = tabnet.predict_proba(X_test_np)

# 3. Création de la figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
colors = plt.cm.get_cmap('tab10')(np.linspace(0, 1, n_classes))

for i, color in zip(range(n_classes), colors):
    # Courbe ROC
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color=color, lw=2,
             label=f'{classes_names[i]} (AUC = {roc_auc:.2f})')

    # Courbe Precision-Recal
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_score[:, i])
    avg_prec = average_precision_score(y_test_bin[:, i], y_score[:, i])
    ax2.plot(recall, precision, color=color, lw=2,
             label=f'{classes_names[i]} (AP = {avg_prec:.2f})')

# Configuration Graphique 1 : ROC
ax1.plot([0, 1], [0, 1], 'k--', lw=2)
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('Taux de Faux Positifs (FPR)')
ax1.set_ylabel('Taux de Vrais Positifs (Recall)')
ax1.set_title('Courbes ROC - TabNet')
ax1.legend(loc="lower right", fontsize='small')
ax1.grid(alpha=0.3)

# Configuration Graphique 2 : Precision-Recall
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Rappel (Recall)')
ax2.set_ylabel('Précision')
ax2.set_title('Courbes Precision-Recall - TabNet')
ax2.legend(loc="lower left", fontsize='small')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()